## memo

- URL: https://www.kaggle.com/code/koyamaryuji/qwen-array-task-inference/log?scriptVersionId=349255171
- v5_id
- answer only adaper

In [6]:
import polars as pl
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent.parent))
from src.gen_task import RAW_RULES, RAW_OOD_RULES_V1

# RULE_CONFIG = RAW_RULES
RULE_CONFIG = RAW_OOD_RULES_V1

RULES = [
    {
        "id": f"{i:03d}",
        "primitives": rule,
    }
    for i, rule in enumerate(RULE_CONFIG)
]

df = pl.read_csv(Path("debug_predictions.csv"))
df = df.sort("id")
stop = 0
for pred in df.iter_rows(named=True):
    print("###" * 50)
    print(f'Task ID: \n{pred["id"].encode().decode("unicode-escape")}')
    print("===" * 50)
    print(f'Prompt: \n{pred["raw_prompt"].encode().decode("unicode-escape")}')
    print("===" * 50)
    print(f'Raw_output: \n{pred["raw_output"]}')
    print("===" * 50)
    print(f'Target: \n{pred["target"]}')
    print("===" * 50)
    print(f'Finish: \n{pred["finish_reason"]}')
    print("===" * 50)
    print(f'tokens: \n{pred["num_tokens"]}')
    if pred["finish_reason"] == "stop":
        stop += 1

print(stop)


######################################################################################################################################################
Task ID: 
000_0
Prompt: 

Infer the transformation rule from examples.
Output the final array.
        
Example 1

Input:
[1, 0, 4, 3, 3, 2, 1, 8, 1, 9]

Output:
[0, 9, 1, 0, 4, 3, 3, 2, 1, 8, 1]

Example 2

Input:
[0, 0, 1, 3, 3, 8, 9, 0]

Output:
[0, 0, 0, 0, 1, 3, 3, 8, 9]

Example 3

Input:
[3, 8, 6, 3, 7, 9, 4, 0, 2]

Output:
[0, 2, 3, 8, 6, 3, 7, 9, 4, 0]

Example 4

Input:
[6, 5, 4, 2, 3, 5, 1, 1, 6, 1]

Output:
[0, 1, 6, 5, 4, 2, 3, 5, 1, 1, 6]

Example 5

Input:
[5, 9, 4, 0, 7, 8, 1]

Output:
[0, 1, 5, 9, 4, 0, 7, 8]

Query

Input:
[1, 8, 4, 9, 5, 9, 3, 1]

Output:
Raw_output: 
<think>

</think>

[0, 1, 1, 8, 4, 9, 5, 9, 3]
Target: 
[0, 1, 1, 8, 4, 9, 5, 9, 3]
Finish: 
stop
tokens: 
32
#################################################################################################################################################

In [7]:
import re
import ast
import polars as pl
from pathlib import Path


def extract_answer(text):
    if text is None:
        return 'NOT_FOUND'

    matches = re.findall(r'\[[^\[\]]*\]', text)
    arrays = []
    for match in matches:
        try:
            value = ast.literal_eval(match)

            if isinstance(value, list):
                arrays.append(value)

        except (ValueError, SyntaxError):
            pass

    if arrays == []:
        return 'NOT_FOUND'

    return str(arrays[-1])

df = pl.read_csv(Path("debug_predictions.csv"))

match_count = 0
match = []
miss = []
for pred in df.iter_rows(named=True):
    # print("###" * 50)
    answer = extract_answer(pred["raw_output"])
    if answer == pred["target"]:
        # print(pred["id"])
        match_count += 1
        match.append(pred["id"].split("_")[0])
    else:
        miss.append(pred["id"].split("_")[0])
        # print(pred["id"])
        # print(pred["raw_prompt"].encode().decode("unicode-escape"))
        # print(pred["target"])
print(f"len(df): {len(df)}")
print(f"match_count: {match_count}")
print(f"acc: {match_count / len(df)}")
# print(miss)
from collections import Counter


counts = Counter(miss)
match_counts = Counter(match)

len(df): 450
match_count: 179
acc: 0.3977777777777778


# 正解

In [8]:
match_results = []

for rule in RULES:
    if rule["id"] in [i for i, j in match_counts.items()]:
        for i, j in match_counts.items():
            if rule["id"] == i:
                match_results.append({"task_id": i, "count": j, "rule": rule["primitives"]})
    else:
        match_results.append({"task_id": rule["id"], "count": 0, "rule": rule["primitives"]})

match_results = sorted(match_results, key=lambda x: x["task_id"], reverse=True)

for x in match_results:
    print(x)

records = []
for x in match_results:
    records.append({"task_id": x["task_id"], "rule": x["rule"], "count": x["count"]})
    print(x)

# heatmap_df = pl.DataFrame(records)
# pl.Config.set_tbl_rows(-1)

# heatmap_df

{'task_id': '044', 'count': 5, 'rule': ['swap_first_last', 'take_odd_positions']}
{'task_id': '043', 'count': 1, 'rule': ['swap_first_last', 'take_even_positions']}
{'task_id': '042', 'count': 5, 'rule': ['swap_first_last', 'mirror']}
{'task_id': '041', 'count': 1, 'rule': ['swap_first_last', 'adjacent_sum']}
{'task_id': '040', 'count': 0, 'rule': ['swap_first_last', 'subtract_next']}
{'task_id': '039', 'count': 1, 'rule': ['swap_first_last', 'mod_3']}
{'task_id': '038', 'count': 0, 'rule': ['swap_first_last', 'mod_2']}
{'task_id': '037', 'count': 0, 'rule': ['swap_first_last', 'add_3']}
{'task_id': '036', 'count': 0, 'rule': ['swap_first_last', 'add_2']}
{'task_id': '035', 'count': 2, 'rule': ['swap_first_last', 'add_1']}
{'task_id': '034', 'count': 1, 'rule': ['swap_first_last', 'multiply_3']}
{'task_id': '033', 'count': 3, 'rule': ['swap_first_last', 'multiply_2']}
{'task_id': '032', 'count': 10, 'rule': ['swap_first_last', 'pop_left']}
{'task_id': '031', 'count': 9, 'rule': ['swap_

# トークン数

In [9]:
print(max(pred["num_tokens"] for pred in df.to_dicts()))

print(next(pred["id"] for pred in df.to_dicts() if pred["num_tokens"] == 65))

65
012_5


# 処理単体と二つの処理の組み合わせの正解率

In [10]:
single_total = 0
single_score = 0
composed_total = 0
composed_score = 0

for pred in df.iter_rows(named=True):
    # print("###" * 50)
    answer = extract_answer(pred["raw_output"])
    if int(pred["id"].split("_")[0]) <= 23:
        single_total += 1
        if answer == pred["target"]:
            # print(pred["id"])
            single_score += 1
    else:
        composed_total += 1
        if answer == pred["target"]:
            # print(pred["id"])
            composed_score += 1
print(f"{single_total}問", single_score, single_score/single_total)
print(f"{composed_total}問", composed_score, composed_score/composed_total)

240問 112 0.4666666666666667
210問 67 0.319047619047619
